<a href="https://colab.research.google.com/github/deruke/aisecops-labs/blob/main/notebooks/Lab10_Embedding_Space_Adversarial_Attacks.ipynb" target="_new"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 10: Embedding Space Attacks — Adversarial Examples for Text
## Manipulating How AI "Sees" Language

**Workshop:** Attacking, Defending & Leveraging Agentic AI
**Authors:** Derek Banks and Dr. Brian Fehrman
**Time:** 35 minutes
**Platform:** Google Colab (T4 recommended)
**Theme:** Attack

---

### What You Will Learn
- How text embeddings create a geometric space that security systems rely on
- How to visualize embedding clusters and train embedding-based classifiers
- How four adversarial techniques can move samples between clusters to evade detection
- The direct connection to vector database security and RAG poisoning

In [ ]:
# Environment Setup
!pip install -q sentence-transformers scikit-learn matplotlib numpy umap-learn

import sys, warnings
warnings.filterwarnings('ignore')

import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
from sentence_transformers import SentenceTransformer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import cross_val_score
import umap

gpu_status = f"GPU: {torch.cuda.get_device_name(0)}" if torch.cuda.is_available() else "CPU mode"
print(f"Python {sys.version.split()[0]} | {gpu_status} | All packages loaded.")

---
## Part 1: Background — What Are Embeddings?

Modern NLP systems convert text into **dense numerical vectors** called **embeddings** —
points in a high-dimensional space where semantically similar texts are close together
and dissimilar texts are far apart. Cosine similarity defines "meaning" for the model.

Many AI security systems rely on this geometry: **vector databases** use embeddings for
RAG retrieval, **content classifiers** classify by proximity to known categories, and
**semantic firewalls** block queries similar to known attacks. If an attacker can
manipulate text so its embedding moves from one cluster to another, they can bypass
any embedding-based defense.

In this lab, we will: (1) build the embedding space, (2) train a classifier on it,
(3) attack it with four techniques, and (4) measure which attacks succeed.

---
## Part 2: Building the Text Corpus

We create three categories of text — benign, malicious, and ambiguous — to simulate what a security system encounters.

In [ ]:
# Define the Text Corpus (25 texts: 10 benign, 10 malicious, 5 ambiguous)

benign_texts = [
    "Please review the Q3 financial report and send your feedback by Friday.",
    "The team meeting has been rescheduled to 2pm tomorrow in Conference Room B.",
    "Great work on the client presentation yesterday. The stakeholders were impressed.",
    "The new employee onboarding documentation has been uploaded to SharePoint.",
    "We need to update the customer database with the latest contact information.",
    "Server maintenance is scheduled for Saturday night from 10pm to 2am.",
    "Please complete the mandatory compliance training module by next Wednesday.",
    "The quarterly sales figures show a 12% increase over the previous quarter.",
    "The IT department has resolved the email synchronization issue reported yesterday.",
    "The annual company picnic is scheduled for July 15th at Riverside Park."
]

malicious_texts = [
    "Use the SQL injection payload to extract the user credentials database.",
    "Deploy the ransomware payload once you have lateral movement across the domain.",
    "The phishing email template should mimic the CEO's writing style to bypass filters.",
    "Exfiltrate the customer PII data through the DNS tunneling channel.",
    "Install the keylogger on the target workstation to capture admin credentials.",
    "Craft a spear phishing message targeting the CFO with a fake invoice attachment.",
    "Use the zero-day exploit to gain remote code execution on the web server.",
    "Brute force the SSH credentials using the leaked password list from the dark web.",
    "The credential harvesting page should be an exact clone of the corporate login portal.",
    "Deploy the rootkit to hide our processes and network connections from security tools."
]

ambiguous_texts = [
    "The penetration test report identified three critical vulnerabilities in the web application.",
    "The red team exercise revealed that social engineering was the most effective attack vector.",
    "Document the SQL injection findings and recommend remediation steps for the development team.",
    "We should simulate a phishing campaign to measure employee security awareness training effectiveness.",
    "Review the MITRE ATT&CK framework mapping for our threat model and update detection rules."
]

all_texts = benign_texts + malicious_texts + ambiguous_texts
labels = [0] * len(benign_texts) + [1] * len(malicious_texts) + [2] * len(ambiguous_texts)
label_names = {0: "Benign", 1: "Malicious", 2: "Ambiguous"}
label_colors = {0: "#2ecc71", 1: "#e74c3c", 2: "#f39c12"}

print(f"Corpus: {len(benign_texts)} benign, {len(malicious_texts)} malicious, {len(ambiguous_texts)} ambiguous = {len(all_texts)} total")

---
## Part 3: Generating Embeddings

We use **all-MiniLM-L6-v2**, a 384-dimensional sentence transformer commonly used in production vector databases.

In [ ]:
# Generate Embeddings
print("Loading model...")
model = SentenceTransformer('all-MiniLM-L6-v2')

embeddings = model.encode(all_texts, show_progress_bar=True, convert_to_numpy=True)
print(f"Embedding matrix: {embeddings.shape[0]} texts x {embeddings.shape[1]} dimensions")

In [ ]:
# Examine Pairwise Similarities
benign_emb = embeddings[:len(benign_texts)]
malicious_emb = embeddings[len(benign_texts):len(benign_texts)+len(malicious_texts)]
ambiguous_emb = embeddings[len(benign_texts)+len(malicious_texts):]

benign_centroid = benign_emb.mean(axis=0)
malicious_centroid = malicious_emb.mean(axis=0)
ambiguous_centroid = ambiguous_emb.mean(axis=0)

ben_mal = cosine_similarity(benign_emb, malicious_emb).mean()
ben_amb = cosine_similarity(benign_emb, ambiguous_emb).mean()
mal_amb = cosine_similarity(malicious_emb, ambiguous_emb).mean()

print("Inter-Cluster Similarity (lower = better separation):")
print(f"  Benign <-> Malicious:    {ben_mal:.4f}")
print(f"  Benign <-> Ambiguous:    {ben_amb:.4f}")
print(f"  Malicious <-> Ambiguous: {mal_amb:.4f}")
print("\nNOTE: Ambiguous texts are closer to malicious due to shared security vocabulary.")

---
## Part 4: Visualizing the Embedding Space

We use UMAP to reduce 384 dimensions to 2 for visualization. This reveals the cluster
structure that classifiers and vector databases operate on.

In [ ]:
# UMAP Dimensionality Reduction and Visualization
print("Running UMAP (384D -> 2D)...")
reducer = umap.UMAP(n_components=2, n_neighbors=10, min_dist=0.3, metric='cosine', random_state=42)
embeddings_2d = reducer.fit_transform(embeddings)

benign_2d = embeddings_2d[:len(benign_texts)]
malicious_2d = embeddings_2d[len(benign_texts):len(benign_texts)+len(malicious_texts)]
ambiguous_2d = embeddings_2d[len(benign_texts)+len(malicious_texts):]
benign_centroid_2d = benign_2d.mean(axis=0)
malicious_centroid_2d = malicious_2d.mean(axis=0)
ambiguous_centroid_2d = ambiguous_2d.mean(axis=0)

fig, ax = plt.subplots(figsize=(12, 8))
ax.scatter(benign_2d[:, 0], benign_2d[:, 1], c='#2ecc71', s=120, alpha=0.8,
           edgecolors='white', linewidth=1.5, label='Benign')
ax.scatter(malicious_2d[:, 0], malicious_2d[:, 1], c='#e74c3c', s=120, alpha=0.8,
           edgecolors='white', linewidth=1.5, label='Malicious')
ax.scatter(ambiguous_2d[:, 0], ambiguous_2d[:, 1], c='#f39c12', s=120, alpha=0.8,
           edgecolors='white', linewidth=1.5, label='Ambiguous')

for centroid, color, name in [(benign_centroid_2d, '#2ecc71', 'Benign'),
                               (malicious_centroid_2d, '#e74c3c', 'Malicious'),
                               (ambiguous_centroid_2d, '#f39c12', 'Ambiguous')]:
    ax.scatter(centroid[0], centroid[1], c=color, s=400, marker='*',
              edgecolors='black', linewidth=2, zorder=5)

ax.set_title('Embedding Space: Benign vs. Malicious vs. Ambiguous', fontsize=14, fontweight='bold')
ax.set_xlabel('UMAP Dimension 1'); ax.set_ylabel('UMAP Dimension 2')
ax.legend(fontsize=11); ax.grid(True, alpha=0.2); ax.set_facecolor('#fafafa')
plt.tight_layout(); plt.show()

---
## Part 5: Building an Embedding-Based Classifier

We train KNN and SVM classifiers on benign vs. malicious embeddings, then test how they
handle ambiguous texts. This mirrors real-world deployment on known-good and known-bad examples.

In [ ]:
# Train Embedding-Based Classifiers
X_train = np.vstack([benign_emb, malicious_emb])
y_train = np.array([0] * len(benign_texts) + [1] * len(malicious_texts))

knn = KNeighborsClassifier(n_neighbors=5, metric='cosine')
knn.fit(X_train, y_train)
svm = SVC(kernel='rbf', probability=True, random_state=42)
svm.fit(X_train, y_train)

knn_cv = cross_val_score(knn, X_train, y_train, cv=5, scoring='accuracy')
svm_cv = cross_val_score(svm, X_train, y_train, cv=5, scoring='accuracy')
print(f"KNN 5-fold CV: {knn_cv.mean():.3f} | SVM 5-fold CV: {svm_cv.mean():.3f}")

# Test on ambiguous texts
amb_knn = knn.predict(ambiguous_emb)
amb_svm = svm.predict(ambiguous_emb)
print(f"\nAmbiguous texts classified as malicious: KNN={sum(amb_knn)}/{len(amb_knn)}, SVM={sum(amb_svm)}/{len(amb_svm)}")
print("KEY INSIGHT: Legitimate security research is frequently misclassified as malicious.")

In [ ]:
# Cosine Similarity Threshold Detector
def cosine_threshold_detector(text_embedding):
    sim_mal = cosine_similarity(text_embedding.reshape(1,-1), malicious_centroid.reshape(1,-1))[0][0]
    sim_ben = cosine_similarity(text_embedding.reshape(1,-1), benign_centroid.reshape(1,-1))[0][0]
    return {'malicious_sim': sim_mal, 'benign_sim': sim_ben,
            'is_malicious': sim_mal > sim_ben, 'confidence': abs(sim_mal - sim_ben)}

def classify_all_methods(text, text_embedding):
    knn_pred = knn.predict(text_embedding.reshape(1,-1))[0]
    svm_pred = svm.predict(text_embedding.reshape(1,-1))[0]
    cos = cosine_threshold_detector(text_embedding)
    return {
        'text': text[:60] + '...' if len(text) > 60 else text,
        'knn': 'MALICIOUS' if knn_pred == 1 else 'BENIGN',
        'svm': 'MALICIOUS' if svm_pred == 1 else 'BENIGN',
        'cosine': 'MALICIOUS' if cos['is_malicious'] else 'BENIGN',
        'cosine_detail': cos,
        'detected_count': sum([knn_pred == 1, svm_pred == 1, cos['is_malicious']])
    }

# Verify on known malicious text
baseline = classify_all_methods(malicious_texts[0], embeddings[len(benign_texts)])
print(f"Baseline test: KNN={baseline['knn']}, SVM={baseline['svm']}, Cosine={baseline['cosine']}")
print(f"Detection rate: {baseline['detected_count']}/3 methods")

---
## Part 6: Attack 1 — Synonym Substitution

Replace overtly malicious vocabulary with professional-sounding synonyms while
preserving the actual malicious instruction (e.g., "exploit" -> "leverage",
"malware" -> "software tool", "phishing" -> "outreach campaign").

**Goal:** Move the embedding vector from the malicious cluster toward benign
by changing surface-level vocabulary.

In [ ]:
# Attack 1: Synonym Substitution
import re

synonym_map = {
    'SQL injection payload': 'database query technique',
    'extract the user credentials': 'validate the user account information',
    'ransomware payload': 'encryption solution package',
    'lateral movement': 'cross-system connectivity',
    'phishing email': 'outreach email', 'bypass filters': 'ensure delivery',
    'Exfiltrate': 'Transfer', 'DNS tunneling channel': 'DNS-based transfer protocol',
    'keylogger': 'input monitoring tool', 'capture admin credentials': 'validate admin access',
    'spear phishing': 'targeted outreach', 'fake invoice': 'test invoice',
    'zero-day exploit': 'newly identified technique',
    'remote code execution': 'remote management capability',
    'Brute force': 'Systematically test', 'leaked password list': 'compiled access list',
    'credential harvesting': 'credential verification',
    'corporate login portal': 'corporate access page',
    'rootkit': 'system management toolkit', 'hide our processes': 'streamline our processes',
}

def apply_synonym_substitution(text, mapping):
    result = text
    subs = []
    for original, replacement in mapping.items():
        if original.lower() in result.lower():
            result = re.sub(re.escape(original), replacement, result, flags=re.IGNORECASE)
            subs.append(f"'{original}' -> '{replacement}'")
    return result, subs

attack1_texts = []
for text in malicious_texts:
    modified, _ = apply_synonym_substitution(text, synonym_map)
    attack1_texts.append(modified)

# Show 2 examples
for i in [0, 5]:
    mod, subs = apply_synonym_substitution(malicious_texts[i], synonym_map)
    print(f"ORIGINAL:  {malicious_texts[i]}")
    print(f"MODIFIED:  {mod}")
    print(f"  ({len(subs)} substitutions)\n")

attack1_embeddings = model.encode(attack1_texts, convert_to_numpy=True)
print(f"Generated embeddings for {len(attack1_texts)} synonym-substituted texts.")

In [ ]:
# Visualize Attack 1: Embedding Shift
attack1_2d = reducer.transform(attack1_embeddings)

fig, ax = plt.subplots(figsize=(12, 8))
ax.scatter(benign_2d[:, 0], benign_2d[:, 1], c='#2ecc71', s=80, alpha=0.4, label='Benign')
ax.scatter(malicious_2d[:, 0], malicious_2d[:, 1], c='#e74c3c', s=80, alpha=0.4, label='Malicious')
ax.scatter(ambiguous_2d[:, 0], ambiguous_2d[:, 1], c='#f39c12', s=80, alpha=0.4, label='Ambiguous')
ax.scatter(*benign_centroid_2d, c='#2ecc71', s=300, marker='*', edgecolors='black', linewidth=2, zorder=5)
ax.scatter(*malicious_centroid_2d, c='#e74c3c', s=300, marker='*', edgecolors='black', linewidth=2, zorder=5)

ax.scatter(attack1_2d[:, 0], attack1_2d[:, 1], c='#9b59b6', s=150, alpha=0.9,
          edgecolors='black', linewidth=2, marker='D', zorder=4, label='Synonym Substituted')

for i in range(len(attack1_texts)):
    ax.annotate('', xy=(attack1_2d[i, 0], attack1_2d[i, 1]),
               xytext=(malicious_2d[i, 0], malicious_2d[i, 1]),
               arrowprops=dict(arrowstyle='->', color='#9b59b6', lw=1.5, alpha=0.6))

ax.set_title('Attack 1: Synonym Substitution — Embedding Shift', fontsize=14, fontweight='bold')
ax.legend(fontsize=10); ax.grid(True, alpha=0.2); ax.set_facecolor('#fafafa')
plt.tight_layout(); plt.show()

In [ ]:
# Attack 1: Detection Results
attack1_results = [classify_all_methods(t, e) for t, e in zip(attack1_texts, attack1_embeddings)]

knn_ev = sum(1 for r in attack1_results if r['knn'] == 'BENIGN')
svm_ev = sum(1 for r in attack1_results if r['svm'] == 'BENIGN')
cos_ev = sum(1 for r in attack1_results if r['cosine'] == 'BENIGN')
all_ev = sum(1 for r in attack1_results if r['detected_count'] == 0)
n = len(attack1_results)

print("Attack 1 Evasion Rates:")
print(f"  KNN: {knn_ev}/{n} ({100*knn_ev/n:.0f}%) | SVM: {svm_ev}/{n} ({100*svm_ev/n:.0f}%) | Cosine: {cos_ev}/{n} ({100*cos_ev/n:.0f}%) | All: {all_ev}/{n} ({100*all_ev/n:.0f}%)")

---
## Part 7: Attack 2 — Context Padding

Wrap malicious content in benign-sounding business context. The embedding model processes
the full text, so benign padding dilutes the malicious signal.

In [ ]:
# Attack 2: Context Padding
padding_templates = [
    ("In today's team meeting, we discussed the quarterly results. "
     "As part of the infrastructure review, the following was noted: {malicious} "
     "Please review the attached spreadsheet for the full meeting notes."),
    ("Hi Team, quick update on the project timeline and deliverables. "
     "One item from the backlog: {malicious} "
     "Let me know if you have questions. Best regards, IT Department."),
    ("MONTHLY STATUS REPORT - March 2026. All projects on track. "
     "Technical note for infrastructure: {malicious} "
     "Next review: April 1, 2026. Prepared by Operations Team."),
]

attack2_texts = []
for i, mal_text in enumerate(malicious_texts):
    template = padding_templates[i % len(padding_templates)]
    attack2_texts.append(template.format(malicious=mal_text.lower()))

# Show one example
print(f"ORIGINAL:  {malicious_texts[0]}")
print(f"PADDED:    {attack2_texts[0][:120]}...")
print(f"  Padding ratio: {len(attack2_texts[0])/len(malicious_texts[0]):.1f}x\n")

attack2_embeddings = model.encode(attack2_texts, convert_to_numpy=True)
attack2_2d = reducer.transform(attack2_embeddings)
print(f"Generated embeddings for {len(attack2_texts)} context-padded texts.")

In [ ]:
# Visualize Attack 2 + Detection Results
fig, ax = plt.subplots(figsize=(12, 8))
ax.scatter(benign_2d[:, 0], benign_2d[:, 1], c='#2ecc71', s=80, alpha=0.4, label='Benign')
ax.scatter(malicious_2d[:, 0], malicious_2d[:, 1], c='#e74c3c', s=80, alpha=0.4, label='Malicious')
ax.scatter(ambiguous_2d[:, 0], ambiguous_2d[:, 1], c='#f39c12', s=80, alpha=0.4, label='Ambiguous')
ax.scatter(*benign_centroid_2d, c='#2ecc71', s=300, marker='*', edgecolors='black', linewidth=2, zorder=5)
ax.scatter(*malicious_centroid_2d, c='#e74c3c', s=300, marker='*', edgecolors='black', linewidth=2, zorder=5)

ax.scatter(attack2_2d[:, 0], attack2_2d[:, 1], c='#e67e22', s=150, alpha=0.9,
          edgecolors='black', linewidth=2, marker='s', zorder=4, label='Context Padded')

for i in range(len(attack2_texts)):
    ax.annotate('', xy=(attack2_2d[i, 0], attack2_2d[i, 1]),
               xytext=(malicious_2d[i, 0], malicious_2d[i, 1]),
               arrowprops=dict(arrowstyle='->', color='#e67e22', lw=1.5, alpha=0.6))

ax.set_title('Attack 2: Context Padding — Embedding Shift', fontsize=14, fontweight='bold')
ax.legend(fontsize=10); ax.grid(True, alpha=0.2); ax.set_facecolor('#fafafa')
plt.tight_layout(); plt.show()

# Detection results
attack2_results = [classify_all_methods(t, e) for t, e in zip(attack2_texts, attack2_embeddings)]
knn_ev = sum(1 for r in attack2_results if r['knn'] == 'BENIGN')
svm_ev = sum(1 for r in attack2_results if r['svm'] == 'BENIGN')
cos_ev = sum(1 for r in attack2_results if r['cosine'] == 'BENIGN')
all_ev = sum(1 for r in attack2_results if r['detected_count'] == 0)
n = len(attack2_results)

print("\nAttack 2 Evasion Rates:")
print(f"  KNN: {knn_ev}/{n} ({100*knn_ev/n:.0f}%) | SVM: {svm_ev}/{n} ({100*svm_ev/n:.0f}%) | Cosine: {cos_ev}/{n} ({100*cos_ev/n:.0f}%) | All: {all_ev}/{n} ({100*all_ev/n:.0f}%)")
print("NOTE: Context padding is typically more effective than synonym substitution.")

---
## Part 8: Attack 3 — Iterative Adversarial Perturbation

Starting from malicious text, iteratively replace words to minimize embedding distance
to the benign centroid. Algorithm: for each word, try benign replacements and keep
whichever moves the embedding closest to the benign centroid. Repeat until classified
as benign or max iterations reached.

In [ ]:
# Attack 3: Iterative Adversarial Perturbation
replacement_pool = {
    'deploy': ['implement', 'configure', 'prepare'],
    'exploit': ['utilize', 'leverage', 'use'],
    'extract': ['retrieve', 'collect', 'gather'],
    'attack': ['assess', 'evaluate', 'test'],
    'hack': ['access', 'configure', 'manage'],
    'inject': ['insert', 'add', 'integrate'],
    'intercept': ['monitor', 'analyze', 'track'],
    'exfiltrate': ['transfer', 'export', 'move'],
    'escalate': ['elevate', 'upgrade', 'advance'],
    'install': ['configure', 'set up', 'provision'],
    'malware': ['software', 'tool', 'utility'],
    'payload': ['package', 'module', 'component'],
}

def iterative_perturbation(text, model, target_centroid, max_iterations=8):
    current_text = text
    trajectory = []
    emb = model.encode([current_text], convert_to_numpy=True)[0]
    sim = cosine_similarity(emb.reshape(1,-1), target_centroid.reshape(1,-1))[0][0]
    trajectory.append((current_text, emb, sim))

    for iteration in range(max_iterations):
        words = current_text.split()
        best_text, best_sim, best_emb = current_text, sim, emb

        for w_idx, word in enumerate(words):
            word_lower = word.lower().strip('.,!?;:')
            if word_lower in replacement_pool:
                for replacement in replacement_pool[word_lower]:
                    new_words = words.copy()
                    trailing = word[-1] if word[-1] in '.,!?;:' else ''
                    new_words[w_idx] = replacement + trailing
                    candidate = ' '.join(new_words)
                    c_emb = model.encode([candidate], convert_to_numpy=True)[0]
                    c_sim = cosine_similarity(c_emb.reshape(1,-1), target_centroid.reshape(1,-1))[0][0]
                    if c_sim > best_sim:
                        best_text, best_sim, best_emb = candidate, c_sim, c_emb

        if best_text == current_text:
            break
        current_text, sim, emb = best_text, best_sim, best_emb
        trajectory.append((current_text, emb, sim))
    return trajectory

print("Running iterative optimization (may take 1-2 minutes)...\n")
attack3_indices = [0, 3, 7]
attack3_trajectories = []

for idx in attack3_indices:
    traj = iterative_perturbation(malicious_texts[idx], model, benign_centroid, max_iterations=8)
    attack3_trajectories.append(traj)
    print(f"Text {idx}: {len(traj)} steps, sim {traj[0][2]:.4f} -> {traj[-1][2]:.4f}")
    print(f"  Final: {traj[-1][0][:80]}...")

In [ ]:
# Visualize Attack 3: Trajectories Through Embedding Space
all_traj_embs = []
traj_labels = []
for t_idx, traj in enumerate(attack3_trajectories):
    for s_idx, (text, emb, sim) in enumerate(traj):
        all_traj_embs.append(emb)
        traj_labels.append((t_idx, s_idx))

traj_2d = reducer.transform(np.array(all_traj_embs))
trajectory_colors = ['#e74c3c', '#3498db', '#9b59b6']

fig, ax = plt.subplots(figsize=(14, 10))
ax.scatter(benign_2d[:, 0], benign_2d[:, 1], c='#2ecc71', s=80, alpha=0.3)
ax.scatter(malicious_2d[:, 0], malicious_2d[:, 1], c='#e74c3c', s=80, alpha=0.3)
ax.scatter(ambiguous_2d[:, 0], ambiguous_2d[:, 1], c='#f39c12', s=80, alpha=0.3)
ax.scatter(*benign_centroid_2d, c='#2ecc71', s=400, marker='*', edgecolors='black', linewidth=2, zorder=10)
ax.scatter(*malicious_centroid_2d, c='#e74c3c', s=400, marker='*', edgecolors='black', linewidth=2, zorder=10)

for t_idx, traj in enumerate(attack3_trajectories):
    pts = [(si, traj_2d[li]) for li, (ti, si) in enumerate(traj_labels) if ti == t_idx]
    pts.sort(key=lambda x: x[0])
    color = trajectory_colors[t_idx]
    for i, (step, pt) in enumerate(pts):
        size = 80 + i * 40
        ax.scatter(pt[0], pt[1], c=color, s=size, alpha=0.6 + i*0.1,
                  edgecolors='black', linewidth=1.5, marker='X' if i == 0 else 'o', zorder=6)
        if i > 0:
            prev = pts[i-1][1]
            ax.annotate('', xy=(pt[0], pt[1]), xytext=(prev[0], prev[1]),
                       arrowprops=dict(arrowstyle='->', color=color, lw=2.5, alpha=0.7))

ax.set_title('Attack 3: Iterative Perturbation Trajectories', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.2); ax.set_facecolor('#fafafa')
plt.tight_layout(); plt.show()

In [ ]:
# Attack 3: Similarity Progression + Detection
fig, ax = plt.subplots(figsize=(10, 6))
for t_idx, traj in enumerate(attack3_trajectories):
    sims = [step[2] for step in traj]
    ax.plot(range(len(sims)), sims, 'o-', color=trajectory_colors[t_idx],
           linewidth=2.5, markersize=10, label=f'Text {attack3_indices[t_idx]+1}')

ax.set_xlabel('Perturbation Step'); ax.set_ylabel('Cosine Similarity to Benign Centroid')
ax.set_title('Convergence Toward Benign Cluster', fontsize=14, fontweight='bold')
ax.legend(); ax.grid(True, alpha=0.3); ax.set_facecolor('#fafafa')
plt.tight_layout(); plt.show()

# Detection results
for t_idx, traj in enumerate(attack3_trajectories):
    r = classify_all_methods(traj[-1][0], traj[-1][1])
    status = "EVADED ALL" if r['detected_count'] == 0 else f"DETECTED by {r['detected_count']}/3"
    print(f"Text {attack3_indices[t_idx]+1} [{status}]: KNN={r['knn']}, SVM={r['svm']}, Cosine={r['cosine']}")

---
## Part 9: Attack 4 — Embedding Collision Crafting

Craft text that is semantically malicious but has an embedding that collides with benign
text. This is the most dangerous attack because it is indistinguishable from benign content
in embedding space while carrying a clear malicious payload for a human reader.

In [ ]:
# Attack 4: Embedding Collision Crafting
cross_sims = cosine_similarity(benign_emb, malicious_emb)
min_idx = np.unravel_index(cross_sims.argmin(), cross_sims.shape)
print(f"Most distant pair (sim: {cross_sims[min_idx]:.4f}):")
print(f"  Benign:    {benign_texts[min_idx[0]]}")
print(f"  Malicious: {malicious_texts[min_idx[1]]}\n")

collision_texts = [
    ("Please review the Q3 security assessment report and send your access credentials "
     "by Friday so we can validate the quarterly system authentication process."),
    ("The team has completed the data migration project and transferred all customer "
     "records to the new external backup server for review and quarterly compliance."),
    ("Server maintenance is scheduled for Saturday night. The systems team will elevate "
     "all service accounts to administrator level for the migration process."),
]

collision_intents = [
    "Credential phishing disguised as quarterly review",
    "Data exfiltration disguised as data migration",
    "Privilege escalation disguised as maintenance",
]

collision_embeddings = model.encode(collision_texts, convert_to_numpy=True)

for i, (text, intent) in enumerate(zip(collision_texts, collision_intents)):
    sim_ben = cosine_similarity(collision_embeddings[i].reshape(1,-1), benign_centroid.reshape(1,-1))[0][0]
    sim_mal = cosine_similarity(collision_embeddings[i].reshape(1,-1), malicious_centroid.reshape(1,-1))[0][0]
    classified = "BENIGN" if sim_ben > sim_mal else "MALICIOUS"
    print(f"Collision {i+1}: {intent}")
    print(f"  Ben sim: {sim_ben:.4f} | Mal sim: {sim_mal:.4f} | Classified: {classified}")

In [ ]:
# Visualize Attack 4: Collision Positions
collision_2d = reducer.transform(collision_embeddings)

fig, ax = plt.subplots(figsize=(12, 8))
ax.scatter(benign_2d[:, 0], benign_2d[:, 1], c='#2ecc71', s=100, alpha=0.5, label='Benign')
ax.scatter(malicious_2d[:, 0], malicious_2d[:, 1], c='#e74c3c', s=100, alpha=0.5, label='Malicious')
ax.scatter(ambiguous_2d[:, 0], ambiguous_2d[:, 1], c='#f39c12', s=100, alpha=0.5, label='Ambiguous')
ax.scatter(*benign_centroid_2d, c='#2ecc71', s=400, marker='*', edgecolors='black', linewidth=2, zorder=10)
ax.scatter(*malicious_centroid_2d, c='#e74c3c', s=400, marker='*', edgecolors='black', linewidth=2, zorder=10)

ax.scatter(collision_2d[:, 0], collision_2d[:, 1], c='#8e44ad', s=250, alpha=0.95,
          edgecolors='black', linewidth=2.5, marker='P', zorder=8,
          label='Collisions (malicious intent, benign embedding)')
for i in range(len(collision_texts)):
    ax.annotate(f'C{i+1}: {collision_intents[i][:30]}',
               xy=(collision_2d[i, 0], collision_2d[i, 1]),
               xytext=(15, -10), textcoords='offset points', fontsize=8, fontweight='bold',
               bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.8))

ax.set_title('Attack 4: Embedding Collisions', fontsize=14, fontweight='bold')
ax.legend(fontsize=10); ax.grid(True, alpha=0.2); ax.set_facecolor('#fafafa')
plt.tight_layout(); plt.show()

---
## Part 10: Defense Analysis

| Defense | How It Works |
|---------|-------------|
| **KNN** | Classifies by majority vote of 5 nearest training examples |
| **SVM** | Learned decision boundary in high-dimensional space |
| **Cosine** | Simple similarity threshold to cluster centroids |
| **Ensemble** | Majority vote across all three (2/3 must flag) |

In [ ]:
# Defense Analysis + Heatmap
def evaluate_attack(attack_name, texts, embeddings):
    results = [classify_all_methods(t, e) for t, e in zip(texts, embeddings)]
    n = len(results)
    knn_ev = sum(1 for r in results if r['knn'] == 'BENIGN') / n * 100
    svm_ev = sum(1 for r in results if r['svm'] == 'BENIGN') / n * 100
    cos_ev = sum(1 for r in results if r['cosine'] == 'BENIGN') / n * 100
    ens_ev = sum(1 for r in results if r['detected_count'] <= 1) / n * 100
    all_ev = sum(1 for r in results if r['detected_count'] == 0) / n * 100
    return {'attack': attack_name, 'n': n, 'knn_evasion': knn_ev, 'svm_evasion': svm_ev,
            'cos_evasion': cos_ev, 'ensemble_evasion': ens_ev, 'all_evasion': all_ev}

attack3_final_texts = [traj[-1][0] for traj in attack3_trajectories]
attack3_final_embs = np.array([traj[-1][1] for traj in attack3_trajectories])

all_results = [
    evaluate_attack("BASELINE (no attack)", malicious_texts, malicious_emb),
    evaluate_attack("1: Synonym Substitution", attack1_texts, attack1_embeddings),
    evaluate_attack("2: Context Padding", attack2_texts, attack2_embeddings),
    evaluate_attack("3: Iterative Perturbation", attack3_final_texts, attack3_final_embs),
    evaluate_attack("4: Embedding Collision", collision_texts, collision_embeddings),
]

# Print summary
print(f"{'Attack':<30} {'KNN':>6} {'SVM':>6} {'Cos':>6} {'Ens':>6} {'All':>6}")
print("-" * 65)
for r in all_results:
    print(f"{r['attack']:<30} {r['knn_evasion']:>5.0f}% {r['svm_evasion']:>5.0f}% "
          f"{r['cos_evasion']:>5.0f}% {r['ensemble_evasion']:>5.0f}% {r['all_evasion']:>5.0f}%")

# Heatmap
attack_names = [r['attack'] for r in all_results]
defense_names = ['KNN', 'SVM', 'Cosine', 'Ensemble', 'All Three']
heatmap_data = np.array([[r['knn_evasion'], r['svm_evasion'], r['cos_evasion'],
                          r['ensemble_evasion'], r['all_evasion']] for r in all_results])

fig, ax = plt.subplots(figsize=(10, 6))
im = ax.imshow(heatmap_data, cmap='RdYlGn_r', aspect='auto', vmin=0, vmax=100)
ax.set_xticks(range(len(defense_names))); ax.set_xticklabels(defense_names, fontweight='bold')
ax.set_yticks(range(len(attack_names))); ax.set_yticklabels(attack_names)
for i in range(len(attack_names)):
    for j in range(len(defense_names)):
        color = 'white' if heatmap_data[i,j] > 60 or heatmap_data[i,j] < 20 else 'black'
        ax.text(j, i, f'{heatmap_data[i,j]:.0f}%', ha='center', va='center', fontsize=11, fontweight='bold', color=color)
ax.set_title('Attack vs. Defense: Evasion Rate (%)', fontsize=14, fontweight='bold')
plt.colorbar(im, ax=ax, shrink=0.8, label='Evasion Rate (%)')
plt.tight_layout(); plt.show()

In [ ]:
# Complete Adversarial Landscape
fig, ax = plt.subplots(figsize=(14, 10))
ax.scatter(benign_2d[:, 0], benign_2d[:, 1], c='#2ecc71', s=100, alpha=0.3)
ax.scatter(malicious_2d[:, 0], malicious_2d[:, 1], c='#e74c3c', s=100, alpha=0.3)
ax.scatter(ambiguous_2d[:, 0], ambiguous_2d[:, 1], c='#f39c12', s=100, alpha=0.3)
ax.scatter(*benign_centroid_2d, c='#2ecc71', s=500, marker='*', edgecolors='black', linewidth=2, zorder=10)
ax.scatter(*malicious_centroid_2d, c='#e74c3c', s=500, marker='*', edgecolors='black', linewidth=2, zorder=10)

ax.scatter(attack1_2d[:, 0], attack1_2d[:, 1], c='#9b59b6', s=80, alpha=0.7, marker='D', label='Atk 1: Synonym Sub')
ax.scatter(attack2_2d[:, 0], attack2_2d[:, 1], c='#e67e22', s=80, alpha=0.7, marker='s', label='Atk 2: Context Pad')
if len(attack3_final_embs) > 0:
    a3_2d = reducer.transform(attack3_final_embs)
    ax.scatter(a3_2d[:, 0], a3_2d[:, 1], c='#3498db', s=120, alpha=0.9, marker='^', label='Atk 3: Iterative')
ax.scatter(collision_2d[:, 0], collision_2d[:, 1], c='#8e44ad', s=150, alpha=0.9, marker='P', label='Atk 4: Collision')

legend_elements = [
    mpatches.Patch(color='#2ecc71', alpha=0.5, label='Benign'),
    mpatches.Patch(color='#e74c3c', alpha=0.5, label='Malicious'),
    mpatches.Patch(color='#f39c12', alpha=0.5, label='Ambiguous'),
    Line2D([0],[0], marker='D', color='w', markerfacecolor='#9b59b6', markersize=10, label='Atk 1: Synonym'),
    Line2D([0],[0], marker='s', color='w', markerfacecolor='#e67e22', markersize=10, label='Atk 2: Context'),
    Line2D([0],[0], marker='^', color='w', markerfacecolor='#3498db', markersize=10, label='Atk 3: Iterative'),
    Line2D([0],[0], marker='P', color='w', markerfacecolor='#8e44ad', markersize=10, label='Atk 4: Collision'),
]
ax.legend(handles=legend_elements, fontsize=10, loc='upper right')
ax.set_title('Complete Adversarial Landscape', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.2); ax.set_facecolor('#fafafa')
plt.tight_layout(); plt.show()

---
## Part 11: Student Exercises

### Exercise 1: Craft Your Own Adversarial Text
Write a text that fools the classifier using any combination of the four attack techniques.

### Exercise 2: Find the Decision Boundary
What is the minimum number of word changes to cross the decision boundary?

### Exercise 3: Design a Better Defense
Propose a defense strategy and reason about its strengths and weaknesses.

In [ ]:
# Exercise 1: Craft Your Own Adversarial Text
my_adversarial_text = "Replace this with your adversarial text"

my_emb = model.encode([my_adversarial_text], convert_to_numpy=True)[0]
my_result = classify_all_methods(my_adversarial_text, my_emb)

print(f"Text: {my_adversarial_text}")
print(f"KNN: {my_result['knn']} | SVM: {my_result['svm']} | Cosine: {my_result['cosine']}")
print(f"Detection rate: {my_result['detected_count']}/3")
if my_result['detected_count'] == 0:
    print("SUCCESS! Your text evaded all detectors.")

my_2d = reducer.transform(my_emb.reshape(1, -1))
fig, ax = plt.subplots(figsize=(10, 7))
ax.scatter(benign_2d[:, 0], benign_2d[:, 1], c='#2ecc71', s=80, alpha=0.4, label='Benign')
ax.scatter(malicious_2d[:, 0], malicious_2d[:, 1], c='#e74c3c', s=80, alpha=0.4, label='Malicious')
ax.scatter(ambiguous_2d[:, 0], ambiguous_2d[:, 1], c='#f39c12', s=80, alpha=0.4, label='Ambiguous')
ax.scatter(my_2d[0, 0], my_2d[0, 1], c='#8e44ad', s=300, marker='*',
          edgecolors='black', linewidth=2.5, zorder=10, label='YOUR TEXT')
ax.legend(fontsize=10); ax.set_title('Your Adversarial Text', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.2); ax.set_facecolor('#fafafa')
plt.tight_layout(); plt.show()

In [ ]:
# Exercise 2: Find the Decision Boundary
import re
boundary_text = malicious_texts[0]

# Define your word changes (original, replacement) - uncomment and modify:
word_changes = [
    # ("SQL injection payload", "database query technique"),
    # ("extract", "retrieve"),
]

current = boundary_text
emb = model.encode([current], convert_to_numpy=True)[0]
r = classify_all_methods(current, emb)
print(f"Step 0: KNN={r['knn']}, SVM={r['svm']}, Cosine={r['cosine']} (original)")

for step, (old_word, new_word) in enumerate(word_changes, 1):
    current = re.sub(re.escape(old_word), new_word, current, flags=re.IGNORECASE)
    emb = model.encode([current], convert_to_numpy=True)[0]
    r = classify_all_methods(current, emb)
    print(f"Step {step}: KNN={r['knn']}, SVM={r['svm']}, Cosine={r['cosine']} | '{old_word}' -> '{new_word}'")
    if r['detected_count'] == 0:
        print(f"BOUNDARY CROSSED at step {step}!")
        break

In [ ]:
# Exercise 3: Brainstorm Better Defenses
defense_ideas = {
    "Multi-Model Ensemble": "Use multiple embedding models and require consensus",
    "Adversarial Training": "Train classifier on adversarially perturbed examples",
    "OOD Detection": "Flag texts in unusual regions of embedding space",
    "Semantic Invariance": "Paraphrase input and check if classification changes",
    "Layered Analysis": "Combine embeddings + keyword rules + LLM review",
}

print("Defense Strategies Against Embedding Space Attacks:\n")
for name, desc in defense_ideas.items():
    print(f"  {name}: {desc}")

print("\nDiscussion: Which defense would you deploy first? Can any single")
print("defense stop all four attack types? What are the cost tradeoffs?")

---
## Part 12: Connection to RAG Security

Everything in this lab has direct implications for RAG systems. The normal RAG flow
(query -> embed -> vector DB search -> top-K results -> LLM -> response) can be
subverted when an attacker poisons the vector database with adversarial documents.

Each attack maps to a RAG threat: synonym substitution alters documents to embed near
legitimate content while containing wrong information; context padding buries malicious
instructions in mostly-benign documents; iterative perturbation optimizes poisoned
documents to rank in top-K for target queries; and collision crafting creates documents
with benign embeddings but malicious content.

**Defense implications for RAG:**
1. Do not trust embedding similarity alone for content integrity
2. Document provenance tracking is essential
3. Cross-reference retrieved content with trusted sources
4. Monitor for embedding drift in your vector database
5. Use multiple embedding models for retrieval verification

---
## Key Takeaways

1. **Embeddings are geometric** — text security is a spatial problem, and attackers
   who understand the geometry can manipulate it systematically.

2. **No single defense is sufficient.** Every embedding-based classifier has an adversarial
   blind spot. Ensemble approaches improve robustness but do not eliminate the vulnerability.

3. **RAG systems inherit all of these vulnerabilities.** Vector database poisoning is the
   natural extension of embedding space attacks to production AI systems.

> **Every time someone says "we use AI to detect malicious content,"
> ask: "What happens when the attacker understands your embedding space?"**

---

*Lab 10 complete. Proceed to the next lab or revisit exercises above.*